### INITIALIZATION

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

In [2]:
import math
import random

In [3]:
from gensim.models import Word2Vec

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, BatchSampler, Sampler
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True

torch.backends.fp32_precision = "tf32"
torch.backends.cudnn.fp32_precision = "tf32"
torch.backends.cudnn.rnn.fp32_precision = "tf32"

In [5]:
from time import perf_counter

In [6]:
import pandas as pd
import numpy as np

In [7]:
assert torch.cuda.is_available()
device = torch.device("cuda")
print(f"Device: {device}")

Device: cuda


### DATASET

In [8]:
def convert_labels(label):
    if label in {"fake", "hate", "rumor", "unreliable", "conspiracy", "bias", "junksci", "satire"}:
        return 0
    elif label in {"reliable", "political", "clickbait"}:
        return 1
    else:
        return pd.NA

In [9]:
metadata = pd.read_csv("./../data/995,000_rows.csv", usecols=["type"])

In [11]:
converted_labels = metadata["type"].apply(convert_labels)
converted_labels.head()

0    1
1    0
2    0
3    1
4    0
Name: type, dtype: object

In [12]:
del metadata

#### WORD EMBEDDING MODEL

In [13]:
model1 = Word2Vec.load("model1.model")

In [14]:
model1.wv["hello"]

array([-1.2373574 ,  0.2523826 ,  1.1372566 , -1.6835815 , -0.68190324,
       -0.6916349 ,  1.1679527 , -0.78110063,  0.66962284,  0.25351796,
        0.39194983, -0.5097382 , -0.35155195,  0.31117496,  0.32204464,
        0.40454262, -0.9431102 ,  0.6477327 , -0.04841546,  0.02628403,
        0.7920931 , -0.02596866,  0.20399731,  0.14207327, -0.40126356,
        0.54467183,  0.37701672,  1.1129324 , -0.12987243, -0.44911227,
        0.15709515,  0.06630421, -0.6807816 , -1.0892999 , -0.54582983,
       -0.29218963, -0.20928013,  0.31765354,  0.5909922 , -0.7469746 ,
        0.01700483,  0.57256603,  0.13983713,  0.22878228,  1.0707805 ,
        0.76070255,  0.03278318,  0.5292343 ,  1.4500283 , -1.1724187 ,
       -0.5478943 ,  0.15152843, -0.35726264,  0.14778645,  1.4770117 ,
       -0.34025547,  0.5487204 ,  0.5322696 , -1.1624863 ,  0.64561325,
        1.6219087 ,  0.02906079, -0.5387904 ,  0.47136173], dtype=float32)

In [15]:
print(f"Embedding size: {len(model1.wv["hello"])}")

Embedding size: 64


In [16]:
model1.wv.get_index("hello")

5414

In [17]:
def tokenlist_to_tokenindexes(tokens):
    return torch.tensor([model1.wv.get_index(token) for token in tokens if token in model1.wv], dtype=torch.long)

In [18]:
file_chunks = pd.read_csv("./../data/data_stemmed.csv", index_col=0, chunksize=50000) 

In [19]:
temp_chunks = []

In [20]:
for chunk in file_chunks:
    chunk = chunk["content"].apply(lambda s: s.split(" "))
    chunk = chunk.apply(tokenlist_to_tokenindexes)
    temp_chunks.append(chunk)


In [21]:
data = pd.concat(temp_chunks).to_frame()
data.head()

,content
0,"[tensor(616), tensor(13), tensor(198), tensor(..."
1,"[tensor(401), tensor(260), tensor(208), tensor..."
2,"[tensor(215), tensor(33060), tensor(2), tensor..."
3,"[tensor(5508), tensor(34799), tensor(155), ten..."
4,"[tensor(41), tensor(2), tensor(1395), tensor(2..."


In [22]:
data.memory_usage(deep=True)

Index       7959904
content    87558944
dtype: int64

In [23]:
data["type"] = converted_labels
data.head()

,content,type
0,"[tensor(616), tensor(13), tensor(198), tensor(...",1
1,"[tensor(401), tensor(260), tensor(208), tensor...",0
2,"[tensor(215), tensor(33060), tensor(2), tensor...",0
3,"[tensor(5508), tensor(34799), tensor(155), ten...",1
4,"[tensor(41), tensor(2), tensor(1395), tensor(2...",0


In [24]:
print(data.isna().sum())
print(data.shape)

content        0
type       91310
dtype: int64
(994988, 2)


In [25]:
data.dropna(inplace=True)
print(data.isna().sum())
print(data.shape)

content    0
type       0
dtype: int64
(903678, 2)


In [26]:
# GROUP BY BUCKET SIZE
buckets = [0, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
data["cuts"] = pd.cut(data["content"].apply(lambda x: x.shape[0]), bins=buckets, labels=buckets[1:])

In [27]:
for bucket_size, group in data.groupby("cuts", observed=True):
    print(f"Length: {len(group)} Bucketsize: {bucket_size}")

Length: 8557 Bucketsize: 8
Length: 44717 Bucketsize: 16
Length: 68358 Bucketsize: 32
Length: 68219 Bucketsize: 64
Length: 135352 Bucketsize: 128
Length: 169996 Bucketsize: 256
Length: 226137 Bucketsize: 512
Length: 139868 Bucketsize: 1024
Length: 33705 Bucketsize: 2048
Length: 6968 Bucketsize: 4096
Length: 1510 Bucketsize: 8192
Length: 286 Bucketsize: 16384
Length: 5 Bucketsize: 32768


In [28]:
data.reset_index(inplace=True, drop=True)

In [29]:
split_index1 = int(len(data)*0.8)
split_index2 = int(len(data)*0.9)

In [30]:
train_set = data.iloc[:split_index1]
val_set = data.iloc[split_index1:split_index2]
test_set = data.iloc[split_index2:]

In [31]:
print(len(train_set))
print(len(val_set))
print(len(test_set))

722942
90368
90368


In [32]:
class MySampler(Sampler):
    def __init__(self, groups, bucket_to_batch_size):
        self.groups = groups
        self.bucket_to_batch_size = bucket_to_batch_size

    def __iter__(self):
        batches = []
        
        for bucket_size, group_indices in self.groups.indices.items():    
            indices = list(group_indices)
            random.shuffle(indices)

            new_batch_size = self.bucket_to_batch_size[bucket_size]
            for i in range(0, len(indices), new_batch_size):
                batch = indices[i:i+new_batch_size]
                if batch:
                    batches.append(batch)

        random.shuffle(batches)
        
        yield from batches

    def __len__(self):
        return len(self.batches)

In [33]:
class ArticleDataset(Dataset):
    def __init__(self, df):
        self.data = df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        articles = self.data["content"].iloc[idx]
        padded_articles = pad_sequence(articles.tolist(), batch_first=True)
        lengths = torch.tensor([a.size(0) for a in articles], dtype=torch.long).cpu()
        
        labels = self.data["type"].iloc[idx]
        labels_tensor = torch.from_numpy(labels.to_numpy(dtype=np.float32))

        # article_tensor = self.embedding(article)
        # label_tensor = torch.tensor([label], dtype=torch.float32)
        return padded_articles, lengths, labels_tensor

In [34]:
def collate_fn(batch):
    articles, labels = zip(*batch)
    padded_articles = pad_sequence(articles, batch_first=True)
    lengths = [a.size(0) for a in articles]

    padded_labels = pad_sequence(labels, batch_first=True)
    return padded_articles, lengths, padded_labels

#### TRAIN, VALIDATION, TEST DATASETS

In [35]:
mysampler = MySampler(train_set.groupby("cuts", observed=True), {8: 512, 16: 512, 32: 512, 64: 512, 128: 512, 256: 512, 512: 256, 1024: 128, 2048: 64, 4096: 32, 8192: 16, 16384: 8, 32768: 4})

train_dataloader = DataLoader(
    ArticleDataset(train_set), 
    batch_size=1,
    sampler=mysampler,
    num_workers=4,
    collate_fn=None, 
    pin_memory=True,
    persistent_workers=True,
)

In [36]:
buckettosize = {8: 32, 16: 32, 32: 32, 64: 32, 128: 32, 256: 32, 512: 16, 1024: 8, 2048: 4, 4096: 2, 8192: 1, 16384: 1, 32768: 1}
val_sampler = MySampler(val_set.groupby("cuts", observed=True), buckettosize)
val_loader = DataLoader(
    ArticleDataset(val_set), 
    batch_size=1,
    sampler=val_sampler,
    num_workers=4,
    collate_fn=None, 
    pin_memory=True,
    persistent_workers=True,
)

In [37]:
for i, (articles, lengths, labels) in enumerate(train_dataloader):
    if i == 4:
        break
    print(articles.size())

torch.Size([1, 64, 2015])
torch.Size([1, 128, 1020])
torch.Size([1, 16, 6575])
torch.Size([1, 64, 1969])


In [38]:
# first iteration timing
start = perf_counter()
batch = next(iter(train_dataloader))
print(perf_counter() - start)

0.20394453033804893


In [39]:
# second iter time
for i, batch in enumerate(train_dataloader):
    if i == 1:
        start = perf_counter()
    elif i == 2:
        end = perf_counter()
    elif i > 2:
        break
print(end - start)

0.010478483512997627


### NN MODEL

In [40]:
class WordRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, tagset_size=1):
        super(WordRNN, self).__init__()

        self.word_embeddings = nn.Embedding.from_pretrained(torch.tensor(model1.wv.vectors))
        self.word_embeddings.weight.requires_grad = False
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, article_data, article_lengths):
        article_embeddings_tensor = self.word_embeddings(article_data)
        
        packed_data = pack_padded_sequence(article_embeddings_tensor, article_lengths, batch_first=True, enforce_sorted=False)
        output, (h_n, c_n) = self.lstm(article_embeddings_tensor)
        last_hidden = h_n[-1]
        
        logit = self.fc(last_hidden)
        # logit = torch.sigmoid(logit)

        return logit

In [41]:
model = WordRNN(64, 128)
model = model.to("cuda")
model

WordRNN(
  (word_embeddings): Embedding(279528, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [42]:
model = torch.compile(model, mode="max-autotune-no-cudagraphs")
model

OptimizedModule(
  (_orig_mod): WordRNN(
    (word_embeddings): Embedding(279528, 64)
    (lstm): LSTM(64, 128, batch_first=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
  )
)

### TRAINING

In [43]:
from torch.amp import autocast, GradScaler

scaler = GradScaler()

In [44]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [45]:
def train_one_batch(model, batch_articles, article_lengths, batch_labels):
    batch_articles = batch_articles.squeeze(0)
    article_lengths = article_lengths.squeeze(0)
    batch_labels = batch_labels.squeeze(0)

    optimizer.zero_grad(set_to_none=True)

    batch_articles = batch_articles.to(device, non_blocking=True)
    batch_labels = batch_labels.to(device, non_blocking=True)

    with autocast(device_type="cuda"):
        outputs = model(batch_articles, article_lengths).view(batch_labels.size(0))
        loss = criterion(outputs, batch_labels)
    
    scaler.scale(loss).backward()
    # loss.backward()
        
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    # optimizer.step()
        
    scaler.update()

    return loss.item()

In [46]:
epoch_losses = []

In [49]:
start_time = perf_counter()
last_eval_loss = np.inf
epochs_since_last_improvement = 0

for epoch in range(0, 3):
    epoch_total_loss = 0.0
    model.train()
    for index, (batch_articles, article_lengths, batch_labels) in enumerate(train_dataloader):
        batch_loss = train_one_batch(model, batch_articles, article_lengths, batch_labels)
        epoch_total_loss += batch_loss
        
        if index % 250 == 0:
            print(f"{index} ", end="")
            
    epoch_avg_loss = epoch_total_loss / (index + 1)
    
    # EARLY STOPPING
    model.eval()
    eval_total_loss = 0
    
    with torch.no_grad():
        for index, (batch_articles, article_lengths, batch_labels) in enumerate(val_loader):
            batch_articles = batch_articles.squeeze(0)
            article_lengths = article_lengths.squeeze(0)
            batch_labels = batch_labels.squeeze(0)
    
            batch_articles = batch_articles.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
    
            outputs = model(batch_articles, article_lengths).view(batch_labels.size(0))
            batch_loss = criterion(outputs, batch_labels)

            eval_total_loss += batch_loss

    eval_avg_loss = eval_total_loss / (index + 1)
    
        
    end_time = perf_counter()
    
    print(f"\nEpoch {epoch} avg loss: {epoch_avg_loss}. Took {end_time - start_time} seconds.")
    print(f"Epoch {epoch} avg eval loss: {eval_avg_loss}. Changed {eval_avg_loss - last_eval_loss}\n")

    if eval_avg_loss < last_eval_loss:
        epochs_since_last_improvement = 0
    else:
        epochs_since_last_improvement += 1
        print("No improvement")
        if epochs_since_last_improvement == 2:
            print(f"Stopping early at epoch {epoch}")
            break
            
    last_eval_loss = eval_avg_loss
    epoch_losses.append(epoch_avg_loss)
    start_time = end_time

0 250 500 750 1000 1250 1500 1750 2000 2250 2500 2750 3000 
Epoch 0 avg loss: 0.1439339344874241. Took 75.02272276114672 seconds.
Epoch 0 avg eval loss: 0.1489151269197464. Changed -inf
0 250 500 750 1000 1250 1500 1750 2000 2250 2500 2750 3000 
Epoch 1 avg loss: 0.13460527948495116. Took 75.17807406745851 seconds.
Epoch 1 avg eval loss: 0.14555507898330688. Changed -0.003360047936439514
0 250 500 750 1000 1250 1500 1750 2000 2250 2500 2750 3000 
Epoch 2 avg loss: 0.12553633740412468. Took 74.3984633302316 seconds.
Epoch 2 avg eval loss: 0.13877639174461365. Changed -0.006778687238693237


#### AFTER TRAINING

In [44]:
torch.save(model.state_dict(), "model2_25epoch.pth")

In [106]:
model.load_state_dict(torch.load("model2_25epoch.pth"))

<All keys matched successfully>

In [114]:
from sklearn.metrics import f1_score

In [121]:
torch.cuda.empty_cache()

In [120]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for index, (batch_articles, article_lengths, batch_labels) in enumerate(val_loader):
        batch_articles = batch_articles.squeeze(0)
        article_lengths = article_lengths.squeeze(0)
        batch_labels = batch_labels.squeeze(0)

        batch_articles = batch_articles.to(device, non_blocking=True)

        output = model(batch_articles, article_lengths)

        probs = torch.sigmoid(output)
        prediction = (probs > 0.5).long().cpu()
        
        predictions.append(prediction)
        true_labels.append(batch_labels)

In [123]:
score = f1_score(torch.cat(true_labels), torch.cat(predictions))
score

0.9525244019592728